### Imports

In [1]:
# !pip install fastf1
# !pip install xgboost
# !pip install tqdm tqdm-joblib

In [2]:
import numpy as np
import pandas as pd
import fastf1 as f1

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, ParameterGrid
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

import matplotlib.pyplot as plt

import logging
import warnings
logging.getLogger('fastf1').setLevel(logging.WARNING)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

# from tqdm import tqdm
# from tqdm_joblib import tqdm_joblib

seed = 10

### Clean and Process Data
Collecting data from FastF1 API.

In [3]:
f1.Cache.enable_cache("f1_cache")

historicalData = pd.DataFrame()

for year in range(2022, 2025):

    print(f"Loading {year}")
    
    qualiHistorical = f1.get_session(year, "Brazil", "Qualifying")
    # print("loading quali")
    qualiHistorical.load()
    raceHistorical = f1.get_session(year, "Brazil", "Race")
    # print("loading race")
    raceHistorical.load()
    
    qualiData = qualiHistorical.results[["Abbreviation", "Q1", "Q2", "Q3", "Position"]].copy()
    qualiData["Q1"] = qualiData["Q1"].dt.total_seconds()
    qualiData["Q2"] = qualiData["Q2"].dt.total_seconds()
    qualiData["Q3"] = qualiData["Q3"].dt.total_seconds()
    qualiData["finalQualiLap(s)"] = pd.NaT  # initialize with NaT for safety
    qualiData.reset_index(drop=True, inplace=True)
    qualiData.loc[:9, "finalQualiLap(s)"] = qualiData.loc[:9, "Q3"]
    # qualiData.iloc[:10, 4] = qualiData.iloc[:10, 3] # loc or iloc, both work.
    qualiData.loc[10:14, "finalQualiLap(s)"] = qualiData.loc[10:14, "Q2"]
    qualiData.loc[15:, "finalQualiLap(s)"] = qualiData.loc[15:, "Q1"]
    qualiData.drop(["Q1", "Q2", "Q3"], axis=1, inplace=True)
    qualiData.rename(columns={"Abbreviation":"Driver"}, inplace=True)
    qualiData
    print("Quali: ", year)
    # display(qualiData)

    gridPositions = raceHistorical.results[["Abbreviation", "GridPosition", "Time", "Position"]].copy()
    gridPositions.rename(columns={'Position': 'FinishPosition'}, inplace=True)
    gridPositions["Time(s)"] = gridPositions["Time"].dt.total_seconds()
    winner_time = gridPositions["Time(s)"].max()
    gridPositions["TotalTime(s)"] = np.where(
        gridPositions["Time(s)"] == winner_time ,
        gridPositions["Time(s)"],
        np.where(gridPositions["Time(s)"].notna(), winner_time + gridPositions["Time(s)"], np.nan))
    gridPositions["GridPosition"].replace(0.0, 20, inplace=True)
    gridPositions.rename(columns={"Abbreviation":"Driver"}, inplace=True)
    gridPositions.sort_values(by="GridPosition", inplace=True)
    gridPositions.reset_index(drop=True, inplace=True)
    gridPositions.drop(["Time"], axis=1, inplace=True)
    print("Starting Grid: ", year)
    # display(gridPositions)
    
    raceData = raceHistorical.laps[["Driver", "LapTime"]].copy()
    raceData["LapTime(s)"] = raceData["LapTime"].dt.total_seconds()
    avgLapTime = raceData.groupby("Driver")["LapTime(s)"].mean().reset_index()
    avgLapTime.rename(columns={"LapTime(s)": "AvgLapTime(s)"}, inplace=True)
    print("Race: ", year)
    # display(avgLapTime)
    
    df = qualiData.merge(gridPositions, left_on="Driver", right_on="Driver")
    df = df.merge(avgLapTime, left_on="Driver", right_on="Driver")
    df["Position"] = df["Position"].astype(int)
    df["GridPosition"] = df["GridPosition"].astype(int)
    df["finalQualiLap(s)"] = df["finalQualiLap(s)"].astype(float)
    df["AvgLapTime(s)"] = df["AvgLapTime(s)"].astype(float)
    df.dropna(inplace=True)
    print("Processed Full Data: ", year)
    # display(df)

    df["Year"] = year
    historicalData = pd.concat([historicalData, df], ignore_index=True)
    
display(historicalData)

Loading 2022
Quali:  2022
Starting Grid:  2022
Race:  2022
Processed Full Data:  2022
Loading 2023
Quali:  2023
Starting Grid:  2023
Race:  2023
Processed Full Data:  2023
Loading 2024


core        WARNING 	No lap data for driver 23
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 23)


Quali:  2024
Starting Grid:  2024
Race:  2024
Processed Full Data:  2024


,Driver,Position,finalQualiLap(s),GridPosition,FinishPosition,Time(s),TotalTime(s),AvgLapTime(s),Year
0,VER,2,71.877,3,6.0,10.056,5924.100,83.438028,2022
1,RUS,3,72.059,1,1.0,5914.044,5914.044,83.296394,2022
2,SAI,5,72.357,7,3.0,4.051,5918.095,83.353451,2022
3,OCO,6,72.425,16,8.0,18.690,5932.734,83.559634,2022
4,ALO,7,72.504,17,5.0,9.561,5923.605,83.431056,2022
5,HAM,8,72.611,2,2.0,1.529,5915.573,83.317930,2022
6,PER,9,75.601,4,7.0,14.080,5928.124,83.494704,2022
7,ALB,11,71.631,19,15.0,36.016,5950.060,83.803662,2022
8,GAS,12,71.675,10,14.0,31.867,5945.911,83.674803,2022
9,VET,13,71.678,9,11.0,26.183,5940.227,83.665169,2022


### Train an XGBoost Model on Historical Data
Using best Qualifying lap, final Qualifying position, and Starting Grid position to predict average full-race lap time per driver.

Running Cross-Validation to automatically pick the model with the optimal combination of parameters

General Guide for XGBoost Parameters:

| Parameter               | Typical range | What it controls                    | Effect                                                     |
| ----------------------- | ------------- | ----------------------------------- | ---------------------------------------------------------- |
| **n_estimators**        | 200 – 1500    | number of trees                     | More trees → higher capacity (risk of overfit)             |
| **learning_rate (eta)** | 0.01 – 0.3    | step size per tree                  | Lower = slower but more accurate if you raise n_estimators |
| **max_depth**           | 3 – 8         | depth of each tree                  | Higher = more complex interactions                         |
| **subsample**           | 0.6 – 1.0     | row sampling per tree               | < 1.0 adds randomness → prevents overfit                   |
| **colsample_bytree**    | 0.6 – 1.0     | feature sampling per tree           | Same idea but on columns                                   |
| **reg_lambda (L2)**     | 0 – 5         | regularization strength             | Higher → smoother, less overfit                            |
| **min_child_weight**    | 1 – 10        | min sum of instance weight per leaf | Higher → prevents splitting small leaves                   |
| **gamma**               | 0 – 1         | min loss reduction to make a split  | Regularization by pruning                                  |


In [4]:
df = historicalData.copy()

X = historicalData[["finalQualiLap(s)", "Position", "GridPosition"]]
# y = historicalData["AvgLapTime(s)"]
y = historicalData["FinishPosition"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=seed)
# model.fit(X_train, y_train)

model = XGBRegressor(
    n_estimators=900,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=seed,
)
# model = XGBRegressor(
#     n_estimators=100,
#     learning_rate=0.8,
#     max_depth=3,
#     random_state=seed
# )
model.fit(X_train, y_train)

# # Grid Search CV
# param_grid = {
#     # 'max_depth': [3, 4, 5, 6],
#     # 'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     # 'n_estimators': [200, 400, 800, 1200],
#     # 'subsample': [0.8, 0.9, 1.0],
#     # 'colsample_bytree': [0.8, 0.9, 1.0],
#     # 'reg_lambda': [0.5, 1.0, 2.0]

#     # 'max_depth': [3, 4], # 4
#     # 'learning_rate': [0.03, 0.05], # 0.03
#     # 'n_estimators': [600, 1000], # 1000
#     # 'subsample': [0.8, 1.0], # 1
#     # 'colsample_bytree': [0.8, 1.0], # 0.8
#     # 'min_child_weight': [1, 3], # 1
#     # 'reg_lambda': [1.0, 2.0] # 1.0

#     # 'max_depth': [5], # 5
#     # 'learning_rate': [0.01], # 0.01
#     # 'n_estimators': [2200], # 1000, 1400, 1800
#     # 'subsample': [1.0], # 1
#     # 'colsample_bytree': [0.8], # 0.8
#     # 'min_child_weight': [1], # 1
#     # 'reg_lambda': [1.0] # 1.0
    
#     'n_estimators': [100, 150, 200],
#     'max_depth': [8, 25, 42],
#     'learning_rate': [0.1, 0.5, 0.9]
# }

# grid = GridSearchCV(
#     XGBRegressor(device="cuda",random_state=seed), # device="cuda" to use GPU
#     param_grid,
#     scoring='neg_mean_absolute_error',
#     cv=3,
#     verbose=2, # 2 = detailed progress. 1 = minimal info
#     # n_jobs=-1 # uses all cpu cores. DO NOT USE IN GPU MODE
#     n_jobs=1 # for GPU mode
# )
# grid.fit(X_train, y_train)

# print("Best params:", grid.best_params_)
# print("MAE:", -grid.best_score_)

# model = grid.best_estimator_

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=900,
             n_jobs=None, num_parallel_tree=None, ...)

### Gather this year's Qualifying data and final Starting Grid positions
This will be input to the trained model to predict this year's average full-race lap time per driver.

In [5]:
quali2025 = f1.get_session(2025, "Brazil", "Qualifying")
quali2025.load()

core        WARNING 	No lap data for driver 5
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 5)


In [6]:
currentData = quali2025.results[["Abbreviation", "Q1", "Q2", "Q3", "Position"]].copy()
currentData["Q1"] = currentData["Q1"].dt.total_seconds()
currentData["Q2"] = currentData["Q2"].dt.total_seconds()
currentData["Q3"] = currentData["Q3"].dt.total_seconds()
currentData["finalQualiLap(s)"] = pd.NaT  # initialize with NaT for safety
currentData.reset_index(drop=True, inplace=True)
currentData.loc[:9, "finalQualiLap(s)"] = currentData.loc[:9, "Q3"]
currentData.loc[10:14, "finalQualiLap(s)"] = currentData.loc[10:14, "Q2"]
currentData.loc[15:, "finalQualiLap(s)"] = currentData.loc[15:, "Q1"]
currentData.drop(["Q1", "Q2", "Q3"], axis=1, inplace=True)
currentData.rename(columns={"Abbreviation":"Driver"}, inplace=True)

currentGridPosition = pd.DataFrame({
    "Driver": ["NOR", "LEC", "HAM", "RUS", "VER", "ANT", "SAI", "PIA",
               "HAD", "BEA", "TSU", "OCO", "HUL", "ALO", "LAW", "BOR", "ALB", "GAS", "STR", "COL"],
    "GridPosition": [1, 3, 13, 6, 16, 2, 15, 4, 5, 8, 19, 19, 10, 11, 7, 20, 12, 9, 14, 18]
})

currentData = currentData.merge(currentGridPosition, left_on="Driver", right_on="Driver")
currentData.dropna(inplace=True)
currentData["Position"] = currentData["Position"].astype(int)
currentData["finalQualiLap(s)"] = currentData["finalQualiLap(s)"].astype(float)

# display(currentData)

### Final Race Predictions!

In [7]:
predictedFinishPosition = model.predict(currentData[["finalQualiLap(s)", "Position", "GridPosition"]])
yPred = currentData.copy()
yPred["FinishPosition"] = predictedFinishPosition

# Rank drivers by predicted race time
yPred = yPred.sort_values(by="FinishPosition").reset_index()
yPred.rename(columns={"GridPosition":"Starting Position", "Position":"Qualifying Position"}, inplace=True)
yPred["Finishing Position"] = range(1, len(yPred) + 1)
# yPred["Total Race Time"] = pd.to_timedelta((yPred["Predicted Avg Lap Time (s)"]*71), unit='s')

# Print final predictions
print("\n🏁 Predicted 2025 Sao Paulo GP Finishing Order 🏁\n")
print(yPred[["Starting Position", "Driver", "Finishing Position"]])

# Evaluate Model
y_pred = model.predict(X_test)
print(f"\n🔍 Model Error (MAE): {mean_absolute_error(y_test, y_pred).astype(int):} positions")


🏁 Predicted 2025 Sao Paulo GP Finishing Order 🏁

    Starting Position Driver  Finishing Position
0                   1    NOR                   1
1                   2    ANT                   2
2                   6    RUS                   3
3                   4    PIA                   4
4                   5    HAD                   5
5                   7    LAW                   6
6                   3    LEC                   7
7                   8    BEA                   8
8                  13    HAM                   9
9                  16    VER                  10
10                 15    SAI                  11
11                 14    STR                  12
12                  9    GAS                  13
13                 10    HUL                  14
14                 11    ALO                  15
15                 12    ALB                  16
16                 19    TSU                  17
17                 18    COL                  18
18                 

In [8]:
# predictedLapTimes = model.predict(currentData[["finalQualiLap(s)", "Position", "GridPosition"]])
# yPred = currentData.copy()
# yPred["Predicted Avg Lap Time (s)"] = predictedLapTimes

# # Rank drivers by predicted race time
# yPred = yPred.sort_values(by="Predicted Avg Lap Time (s)").reset_index()
# yPred.rename(columns={"GridPosition":"Starting Position"}, inplace=True)
# yPred["Finishing Position"] = range(1, len(yPred) + 1)
# yPred["Total Race Time"] = pd.to_timedelta((yPred["Predicted Avg Lap Time (s)"]*71), unit='s')

# # Print final predictions
# print("\n🏁 Predicted 2025 Sao Paulo GP Finishing Order 🏁\n")
# print(yPred[["Starting Position", "Finishing Position", "Driver"]])

# # Evaluate Model
# y_pred = model.predict(X_test)
# print(f"\n🔍 Model Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} seconds")